# ClipCap: Extended Attention Mask

Mục tiêu:

- Nhận caption `attention_mask` có shape `[B, L]`.
- Thêm `P` vị trí prefix hợp lệ ở đầu sequence.
- Trả về `extended_attention_mask` có shape `[B, P + L]`.
- Giữ nguyên dtype, device và dữ liệu đầu vào.

## 1. Vị trí trong pipeline ClipCap

```text
prefix_embeddings [B, P, D] -----------+
                                         +--> inputs_embeds [B, P + L, D]
text_embeddings [B, L, D] -------------+

prefix mask [B, P] ---------------------+
                                         +--> extended_attention_mask [B, P + L]
caption attention_mask [B, L] ----------+
```

Quy ước:

- `1`: vị trí hợp lệ mà GPT-2 được sử dụng làm ngữ cảnh.
- `0`: vị trí padding cần được bỏ qua.
- Visual prefix chứa thông tin ảnh hợp lệ, nên toàn bộ `P` vị trí prefix có mask bằng `1`.

Attention mask không quyết định vị trí nào được tính loss. Việc đó thuộc trách nhiệm của `extended_labels`, trong đó prefix và padding có label bằng `-100`.

In [ ]:
import torch
from torch import Tensor


def extend_attention_mask(
    attention_mask: Tensor,
    prefix_length: int,
) -> Tensor:
    """Tạo Extended Attention Mask bằng cách nối Prefix Mask (toàn 1) vào trước Caption Attention Mask.

    Args:
        attention_mask (Tensor): Mask của caption [B, L], giá trị 1 cho token thật và 0 cho padding.
        prefix_length (int): Số lượng visual prefix tokens (P).

    Returns:
        Tensor: Extended attention mask có shape [B, P + L], giữ nguyên dtype và device của input.
    """
    if attention_mask.ndim != 2:
        raise ValueError(
            f"attention_mask phải có shape [B, L], nhận được: {tuple(attention_mask.shape)}"
        )

    if (
        isinstance(prefix_length, bool)
        or not isinstance(prefix_length, int)
        or prefix_length <= 0
    ):
        raise ValueError(
            f"prefix_length phải là số nguyên dương > 0, nhận được: {prefix_length}"
        )

    batch_size = attention_mask.size(0)

    # Prefix Mask = 1 cho toàn bộ P vị trí prefix [B, P]
    prefix_mask = torch.ones(
        (batch_size, prefix_length),
        dtype=attention_mask.dtype,
        device=attention_mask.device,
    )

    # Nối Prefix Mask [B, P] và Caption Attention Mask [B, L] -> [B, P + L]
    return torch.cat([prefix_mask, attention_mask], dim=1)

In [ ]:
# Sanity check & Báo cáo chi tiết cho Extended Attention Mask
batch_size = 32
caption_length = 20
prefix_length = 10

# 1. Giả lập caption attention mask (với vài vị trí padding)
test_caption_mask = torch.ones(batch_size, caption_length, dtype=torch.long)
test_caption_mask[:, 15:] = 0  # Giả lập 5 token cuối là padding

# 2. Thực thi hàm
extended_mask = extend_attention_mask(test_caption_mask, prefix_length=prefix_length)

# 3. In Báo cáo Chi tiết
print(f"Input Shape (Caption Mask)     : {tuple(test_caption_mask.shape)}")
print(f"Prefix Length                  : {prefix_length}")
print(f"Output Shape (Extended Mask)   : {tuple(extended_mask.shape)}")
print(f"Prefix Mask Portion Check      : {extended_mask[0, :prefix_length].tolist()}")
print(f"Caption Mask Portion Check     : {extended_mask[0, prefix_length:].tolist()}")

# 4. Kiểm tra Assertion
expected_shape = (batch_size, prefix_length + caption_length)
assert extended_mask.shape == expected_shape, f"ERR: Shape không bằng {expected_shape}"
assert (extended_mask[:, :prefix_length] == 1).all(), "ERR: Prefix mask phải toàn giá trị 1"
assert torch.equal(extended_mask[:, prefix_length:], test_caption_mask), "ERR: Phần caption mask bị sai"

print("PASS: basic sanity check")

## 2. Kiểm tra shape động, dtype và không sửa input

Production code không được phụ thuộc vào một batch size, caption length hoặc prefix length cố định. Các test sau bao phủ nhiều cấu hình và ba dtype attention mask thường gặp: `torch.long`, `torch.bool` và `torch.float32`.

In [ ]:
test_cases = (
    (1, 1, 1),
    (2, 6, 3),
    (4, 9, 10),
)
test_dtypes = (torch.long, torch.bool, torch.float32)

for dtype in test_dtypes:
    for batch_size, caption_length, prefix_length in test_cases:
        caption_mask = torch.ones(
            (batch_size, caption_length),
            dtype=dtype,
        )
        if caption_length > 1:
            caption_mask[:, -1] = 0

        original_mask = caption_mask.clone()
        result = extend_attention_mask(
            caption_mask,
            prefix_length,
        )

        assert result.shape == (
            batch_size,
            prefix_length + caption_length,
        )
        assert result.dtype == caption_mask.dtype
        assert result.device == caption_mask.device
        assert torch.all(result[:, :prefix_length] == 1)
        assert torch.equal(
            result[:, prefix_length:],
            caption_mask,
        )
        assert torch.equal(caption_mask, original_mask)

print("PASS: dynamic shape, dtype, device and input immutability")

## 3. Kiểm tra device

Tensor prefix mask phải được tạo trên cùng device với caption mask. Cell này luôn kiểm tra CPU và tự kiểm tra CUDA nếu máy đang chạy notebook có GPU khả dụng.

In [ ]:
devices = [torch.device("cpu")]
if torch.cuda.is_available():
    devices.append(torch.device("cuda"))

for device in devices:
    caption_mask = torch.tensor(
        [[1, 1, 0], [1, 1, 1]],
        dtype=torch.long,
        device=device,
    )
    result = extend_attention_mask(caption_mask, prefix_length=4)

    assert result.device == caption_mask.device
    assert result.shape == (2, 7)

print("PASS: device checks on", [str(device) for device in devices])

## 4. Kiểm tra input không hợp lệ

Hàm cần báo lỗi rõ ràng khi attention mask không có shape `[B, L]` hoặc `prefix_length` không phải số nguyên dương.

In [ ]:
def expect_error(error_type, function, *args):
    try:
        function(*args)
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")


valid_mask = torch.tensor([[1, 1, 0]], dtype=torch.long)

expect_error(ValueError, extend_attention_mask, valid_mask[0], 2)
for invalid_prefix_length in (0, -1, True, 1.5):
    expect_error(
        ValueError,
        extend_attention_mask,
        valid_mask,
        invalid_prefix_length,
    )

print("PASS: invalid inputs are rejected")

## 5. Integration contract với embeddings và extended labels

Khi tích hợp vào `ClipCaptionModel`, chiều sequence của ba tensor phải bằng nhau:

```text
inputs_embeds             [B, P + L, D]
extended_attention_mask   [B, P + L]
extended_labels           [B, P + L]
```

Nên lấy `prefix_length` từ output thực tế của Mapper bằng `prefix_embeddings.size(1)` thay vì hard-code giá trị `10`.

In [ ]:
batch_size = 2
caption_length = 5
embedding_dim = 32

prefix_embeddings = torch.randn(batch_size, 4, embedding_dim)
text_embeddings = torch.randn(batch_size, caption_length, embedding_dim)
prefix_length = prefix_embeddings.size(1)

caption_mask = torch.tensor(
    [[1, 1, 1, 0, 0], [1, 1, 1, 1, 0]],
    dtype=torch.long,
)
input_ids = torch.tensor(
    [[11, 12, 13, 0, 0], [21, 22, 23, 24, 0]],
    dtype=torch.long,
)

inputs_embeds = torch.cat(
    [prefix_embeddings, text_embeddings],
    dim=1,
)
extended_attention_mask = extend_attention_mask(
    caption_mask,
    prefix_length,
)

caption_labels = input_ids.masked_fill(caption_mask == 0, -100)
prefix_labels = torch.full(
    (batch_size, prefix_length),
    fill_value=-100,
    dtype=input_ids.dtype,
)
extended_labels = torch.cat(
    [prefix_labels, caption_labels],
    dim=1,
)

sequence_length = inputs_embeds.size(1)
assert extended_attention_mask.size(1) == sequence_length
assert extended_labels.size(1) == sequence_length
assert torch.all(extended_attention_mask[:, :prefix_length] == 1)
assert torch.all(extended_labels[:, :prefix_length] == -100)

print("inputs_embeds shape           :", tuple(inputs_embeds.shape))
print("extended_attention_mask shape:", tuple(extended_attention_mask.shape))
print("extended_labels shape        :", tuple(extended_labels.shape))
print("PASS: integration contract")

## 6. Cách gọi trong `ClipCaptionModel.forward()`

```python
prefix_embeddings = self.mapper(image_embed)
prefix_length = prefix_embeddings.size(1)

extended_attention_mask = extend_attention_mask(
    attention_mask=attention_mask,
    prefix_length=prefix_length,
)

outputs = self.gpt2(
    inputs_embeds=inputs_embeds,
    attention_mask=extended_attention_mask,
    labels=extended_labels,
)
```

Caption attention mask vẫn do tokenizer và Dataset cung cấp. Chỉ extended attention mask được tạo trong model vì model biết số visual prefix token thực tế.

## 7. Các lỗi thường gặp

### Đặt prefix mask bằng `0`

GPT-2 sẽ không sử dụng visual prefix đúng cách. Prefix chứa ngữ cảnh ảnh hợp lệ nên mask phải bằng `1`.

### Hard-code `prefix_length = 10` trong model

Nếu cấu hình Mapper thay đổi, mask sẽ lệch chiều. Hãy dùng `prefix_embeddings.size(1)`.

### Tạo prefix mask trên CPU khi caption mask ở GPU

Hai tensor khác device không thể concatenate. Hàm hiện tại đã xử lý đúng bằng `device=attention_mask.device`.

### Nhầm attention mask với labels

Prefix attention mask bằng `1`, nhưng prefix label bằng `-100`. Attention mask điều khiển ngữ cảnh; labels điều khiển loss.

### Tạo lại caption mask từ token ID

GPT-2 thường dùng EOS token làm padding token. Không nên suy luận padding chỉ bằng token ID vì có thể nhầm EOS thật với padding. Hãy giữ attention mask do tokenizer tạo.

## 8. Checklist bàn giao

- [ ] Input attention mask có shape `[B, L]`.
- [ ] Output có shape `[B, P + L]`.
- [ ] `P` vị trí đầu đều bằng `1`.
- [ ] Phần caption mask được giữ nguyên.
- [ ] Dtype và device được giữ nguyên.
- [ ] Không sửa tensor đầu vào.
- [ ] Không hard-code batch size, caption length hoặc prefix length.
- [ ] Sequence length khớp với `inputs_embeds` và `extended_labels`.
- [ ] Chạy được trên CPU và CUDA nếu môi trường có GPU.
- [ ] Invalid input được phát hiện bằng lỗi rõ ràng.